# Trayectoria de un colibrí a partir de un video

Este notebook toma tus videos y, para cada uno, obtiene:

1. **La trayectoria:** dónde está el ave en cada cuadro.
2. **Los puntos clave:** unos pocos puntos que resumen el recorrido, y la curva que pasa por ellos.
3. **La curva 3D (opcional):** esos puntos con una profundidad relativa a la flor y al fondo.

Los resultados quedan en `salidas/<video>/`. Ejecuta las celdas en orden.

**Da mejores resultados con** cámara fija y un ave a la vez. Si hay varias aves se obtiene una sola trayectoria, con los
fragmentos más largos que no se solapan en el tiempo. Si la cámara se mueve, la trayectoria se mueve con ella y la curva 3D no es fiable.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
from colibri.config import CARPETAS, DETECTOR
from colibri.flujo import calcular_profundidad, carpeta_salida, descargar_urls, leer_videos, procesar_3d, procesar_video, ver_cuadro

## 1. Elige los videos

Por defecto se usan todos los de la carpeta `videos/` del repositorio. Si quieres otra cosa, cambia estas opciones:

- `CARPETA`: otra carpeta, o un video concreto.
- `ARCHIVOS`: solo estos archivos de la carpeta, por ejemplo `["visita1.mp4", "visita2.mov"]`.
- `URLS`: enlaces que se descargan a la carpeta antes de empezar, por ejemplo de YouTube o Vimeo. Usa solo videos sobre los que tengas derechos.
- `DETECTOR_A_USAR`: otro detector, por ejemplo uno entrenado con `entrenar_detector.ipynb`.

In [ ]:
CARPETA = CARPETAS["videos"]
ARCHIVOS = None
URLS = []
DETECTOR_A_USAR = DETECTOR

descargar_urls(URLS, CARPETA)
VIDEOS = leer_videos(CARPETA, ARCHIVOS)
display(pd.DataFrame(VIDEOS).T.drop(columns="path").rename_axis("video"))

## 2. Trayectoria y puntos clave

Puede tardar: el detector revisa cada cuadro. Con GPU suele ir bastante más rápido que en tiempo real; sin GPU, mucho más lento.

**Cómo leer el resumen**

| Columna | Qué significa | Qué mirar |
|---|---|---|
| `pct_con_ave` | Porcentaje de cuadros con ave | Si es 0 o muy bajo, el detector no encuentra el ave en ese video |
| `segmentos` | Tramos separados de la trayectoria | Muchos indican salidas de cuadro o detecciones perdidas |
| `puentes` | Huecos cortos rellenados por interpolación | Salen en discontinua en `trayectoria.png` |
| `puentes_largos` | Puentes que cruzan mucha distancia | Comprueba que no estén uniendo dos visitas distintas |
| `error_medio_px` | Distancia media entre el recorrido y la curva de puntos clave | Cuanto menor, mejor resume la curva el recorrido |

No se miden aciertos porque, para tus videos, no hay anotaciones con las que comparar: revisa las figuras.

Si el detector falla con tus videos, puedes ajustarlo con `entrenar_detector.ipynb`. Para cambiar el número de puntos o la forma de
elegirlos, edita `puntos` en `config.yaml`.

In [ ]:
RESUMEN = pd.DataFrame([procesar_video(nombre, video, DETECTOR_A_USAR) for nombre, video in VIDEOS.items()])
RESUMEN.to_csv(CARPETAS["salidas"] / "resumen.csv", index=False)
display(RESUMEN)
LISTOS = RESUMEN[RESUMEN.listo].video.tolist()
for nombre in LISTOS:
    display(Image(filename=str(carpeta_salida(nombre) / "puntos_sobre_video.png"), width=800))

Si solo necesitas la trayectoria 2D, ya has terminado: los resultados están en `salidas/`.

## 3. Curva 3D (opcional)

Un video no tiene profundidad real, así que este paso **la estima** con un modelo y la mide en relación con dos zonas de la propia imagen:

`z = (profundidad del ave − profundidad de la flor) / (profundidad del fondo − profundidad de la flor)`

Así, 0 significa a la altura de la flor, 1 a la distancia del fondo, y un valor negativo, delante de la flor. **No es una distancia
en metros ni una medida validada:** tómala como una exploración.

### Marca la flor y el fondo

Necesitas indicar dos rectángulos por video, con coordenadas de 0 a 1 (`x0, y0, x1, y1`), que puedes leer en la rejilla de la siguiente celda:

- **flor:** lo que visita el ave;
- **fondo:** una zona lejana y estable por donde el ave no pase.

Los videos sin rectángulos no pasan por este paso.

In [ ]:
for nombre in LISTOS:
    ver_cuadro(nombre, VIDEOS[nombre])

Escríbelos aquí, con el nombre de video que aparece en el título de cada imagen:

```python
REGIONES = {
    "mi_video": {"flor": (0.54, 0.38, 0.69, 0.58), "fondo": (0.30, 0.03, 0.62, 0.16)},
}
```

In [ ]:
REGIONES = {
}

### Calcula la profundidad

La primera vez se descarga el modelo. Después, cada video tarda unos minutos, porque se procesa completo dos veces, y necesita una GPU NVIDIA:
un video de 20 segundos usó unos 3 GB de memoria. Si no cabe, la resolución baja sola y lo verás en el mensaje; **compara solo resultados
con la misma resolución**, porque influye en la lectura. Empieza por clips cortos.

In [ ]:
SIN_REGIONES = [n for n in LISTOS if n not in REGIONES]
if SIN_REGIONES:
    print("Sin regiones, no pasan por este paso:", SIN_REGIONES)
CON_REGIONES = {n: VIDEOS[n] for n in LISTOS if n in REGIONES}
if CON_REGIONES:
    cache = calcular_profundidad(CON_REGIONES)
    CONTROL = pd.DataFrame([procesar_3d(n, v, REGIONES[n], cache) for n, v in CON_REGIONES.items()])
    CONTROL.to_csv(CARPETAS["salidas"] / "control_3d.csv", index=False)
    display(CONTROL.round(3))

### Cómo revisar el resultado

- `forma_supera_ruido`: con `True`, la diferencia entre puntos es mayor que la variación entre cuadros vecinos, así que la forma en z tiene alguna señal. Con `False`, no la interpretes.
- `revision_3d.jpg`: el recorte de cada punto con su valor. Compruébalo a ojo: ¿el ave está de verdad delante o detrás de la flor?
- `trayectoria_3d.html`: abre este archivo en el navegador para girar la curva.